# DataMate (Zalo AI 2026) — chạy trên Kaggle, **Run All** ra file nộp public

**Chuẩn bị 1 lần (bên phải màn hình notebook):**
1. *Settings → Accelerator* = **GPU T4 x2**. *Settings → Internet* = **On** (clone code, cài vLLM, tải model).
2. *Add Input* → dataset `gmnguynhng/copypaste` (dữ liệu train + public_test; thư mục hay file `.zip` đều được).
3. Repo GitHub **private** → *Add-ons → Secrets* thêm `GITHUB_TOKEN` (token chỉ đọc) và bật cho notebook. Repo public thì bỏ qua.
4. Bấm **Run All** (hoặc *Save Version → Save & Run All*).

**Kết quả:** `/kaggle/working/submission.jsonl` (đủ 1 dòng/câu public, đúng thứ tự `questions.jsonl`) + `outputs.zip` (log).

Notebook **không chấm** public (không có và không dùng đáp án public — quy định 6). Chỉ chấm trên TRAIN.
Mọi bước có dự phòng: cài vLLM lỗi → thử bản khác → backend `hf`; model lỗi giữa chừng → vẫn có đáp án của đường máy tính;
sắp hết 12 giờ → tự dừng và ghi file nộp đủ câu.

## 0. Cấu hình (chỉ sửa ô này nếu cần)

In [ ]:
import os, sys, time, json, glob, shutil, zipfile, subprocess, collections, re
T0 = time.time()                       # mốc tính ngân sách thời gian của cả phiên

REPO_URL   = "https://github.com/Hana-ai-18/CopyPaste.git"
BRANCH     = ""                        # "" = nhánh mặc định của repo
INPUT_DIR  = "/kaggle/input/datasets/gmnguynhng/copypaste"
WORK       = "/kaggle/working"

MODEL      = "Qwen/Qwen2.5-Coder-7B-Instruct-AWQ"   # vLLM (T4: AWQ fp16)
MODEL_HF   = "Qwen/Qwen2.5-Coder-7B-Instruct"       # chỉ dùng nếu phải lùi về backend hf (4-bit bitsandbytes, CHẬM)
MAX_LEN    = 28000                     # đủ cho rulebook + thẻ dữ liệu + câu hỏi; prompt dài hơn tự bỏ qua câu đó
N_PATHS    = 2                         # 2 đường độc lập, chỉ trả lời khi đồng thuận
SESSION_HOURS = 11.5                   # Kaggle cắt phiên ở 12 giờ: để dư 30 phút
RUN_SMOKE  = True                      # đo nhanh trên vài câu TRAIN (có điểm) trước khi chạy public
RUN_TRAIN_EVAL = False                 # True: chạy 6 shop TRAIN + fit mô hình bỏ trống (thêm ~1–2 giờ GPU) rồi mới chạy public

# chế độ thử cục bộ (không dùng trên Kaggle)
TEST_MODE = os.environ.get("DM_NB_TEST") == "1"
if TEST_MODE:
    REPO_URL  = os.environ["DM_REPO"]
    INPUT_DIR = os.environ["DM_INPUT"]
    WORK      = os.environ["DM_WORK"]
os.makedirs(WORK, exist_ok=True)
DATA, CACHE = f"{WORK}/data", f"{WORK}/cache"
os.makedirs(DATA, exist_ok=True); os.makedirs(CACHE, exist_ok=True)

def elapsed_h():
    return (time.time() - T0) / 3600

def sh(cmd, check=True, timeout=None, env=None, quiet=False):
    """Chạy lệnh shell, in log theo thời gian thực, trả (mã thoát, 60 dòng cuối). check=True: lỗi thì dừng notebook."""
    print(f"$ {cmd if len(cmd) < 400 else cmd[:400] + ' …'}", flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, **(env or {})}, bufsize=1)
    tail = collections.deque(maxlen=60)
    t = time.time()
    try:
        for line in p.stdout:
            tail.append(line.rstrip("\n"))
            if not quiet:
                print(line, end="", flush=True)
            if timeout and time.time() - t > timeout:
                p.kill(); tail.append(f"[hết {timeout}s → dừng]"); break
        p.wait()
    except KeyboardInterrupt:
        p.kill(); raise
    if quiet and p.returncode:
        print("\n".join(tail))
    if check and p.returncode:
        raise RuntimeError(f"Lệnh lỗi (mã {p.returncode}): {cmd[:200]}")
    return p.returncode, "\n".join(tail)

print("Python", sys.version.split()[0], "| TEST_MODE =", TEST_MODE)

## 1. GPU

In [ ]:
rc, out = sh("nvidia-smi --query-gpu=index,name,memory.total,compute_cap --format=csv,noheader", check=False)
GPUS = [l for l in out.splitlines() if l.strip() and "," in l] if rc == 0 else []
N_GPU = len(GPUS)
print("Số GPU:", N_GPU)
if not TEST_MODE and N_GPU == 0:
    print("!!! KHÔNG CÓ GPU: vào Settings → Accelerator → GPU T4 x2 rồi Run All lại. "
          "Notebook vẫn chạy tiếp nhưng chỉ có đường máy tính (không LLM).")

## 2. Dữ liệu: giải nén (nếu là .zip) và tìm thư mục train / public

In [ ]:
if not os.path.exists(INPUT_DIR):
    print("Không thấy", INPUT_DIR, "→ tìm trong toàn bộ /kaggle/input")
    INPUT_DIR = "/kaggle/input"

def unzip_all(src_root, dst):
    for z in sorted(glob.glob(f"{src_root}/**/*.zip", recursive=True)):
        out = os.path.join(dst, os.path.splitext(os.path.basename(z))[0])
        if not os.path.exists(out):
            print("giải nén", z)
            with zipfile.ZipFile(z) as f:
                f.extractall(out)
unzip_all(INPUT_DIR, DATA)
for _ in range(2):                               # zip lồng nhau
    unzip_all(DATA, DATA)

def find_splits():
    found = []
    for root in (INPUT_DIR, DATA):
        for rb in glob.glob(f"{root}/**/rulebook.md", recursive=True):
            d = os.path.dirname(rb)
            if os.path.exists(f"{d}/questions.jsonl") and os.path.isdir(f"{d}/data") and d not in found:
                found.append(d)
    return found

SPLITS = find_splits()
for d in SPLITS:
    print(d, "| có answers.jsonl:", os.path.exists(f"{d}/answers.jsonl"))
# TRAIN = thư mục CÓ đáp án; PUBLIC = thư mục KHÔNG có đáp án, ưu tiên tên chứa 'public'
TRAIN = next((d for d in SPLITS if os.path.exists(f"{d}/answers.jsonl")), None)
no_ans = [d for d in SPLITS if not os.path.exists(f"{d}/answers.jsonl")]
PUBLIC = next((d for d in no_ans if "public" in d.lower()), no_ans[0] if no_ans else None)
print("TRAIN :", TRAIN)
print("PUBLIC:", PUBLIC)
if not (TRAIN and PUBLIC):
    sh(f"find {INPUT_DIR} -maxdepth 4 | head -60", check=False)
    raise SystemExit("Không tìm thấy train (có answers.jsonl) và public (không có). Mỗi split cần rulebook.md + questions.jsonl + data/.")
for d in (TRAIN, PUBLIC):
    nq = sum(1 for _ in open(f"{d}/questions.jsonl", encoding="utf-8"))
    print(f"{d}: {len(os.listdir(d + '/data'))} shop, {nq} câu")

## 3. Lấy code từ GitHub

In [ ]:
CLONE = f"{WORK}/repo"
url = REPO_URL
if not TEST_MODE:
    try:                                             # repo private: token trong Kaggle Secrets
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
        if tok:
            url = REPO_URL.replace("https://", f"https://x-access-token:{tok}@")
            print("Dùng GITHUB_TOKEN từ Kaggle Secrets")
    except Exception:
        print("Không có secret GITHUB_TOKEN → clone như repo public")
shutil.rmtree(CLONE, ignore_errors=True)
rc, out = sh(f"git clone -q --depth 1 {'-b ' + BRANCH if BRANCH else ''} '{url}' '{CLONE}'", check=False, quiet=True)
if rc:
    raise SystemExit("Clone lỗi. Kiểm tra: Internet = On; REPO_URL đúng; repo private thì cần secret GITHUB_TOKEN.\n"
                     + out.replace(url, REPO_URL))
sh(f"git -C '{CLONE}' remote set-url origin '{REPO_URL}'", check=False, quiet=True)   # xoá token khỏi .git/config
# code có thể nằm ở gốc repo hoặc trong thư mục con (vd khi đẩy cả thư mục datamate-pipeline/)
hits = sorted(glob.glob(f"{CLONE}/**/datamate_pipeline.py", recursive=True), key=len)
if not hits:
    raise SystemExit("Repo không có datamate_pipeline.py")
CODE = os.path.dirname(hits[0])
os.chdir(CODE)
sh(f"git -C '{CLONE}' log -1 --format='commit %h  %ad  %s' --date=iso", check=False)   # ghi commit đang chạy
need = ["datamate_pipeline.py", "compiler.py", "ledger.py", "rule_delta.py", "retriever.py",
        "lexicon/rulebook_train_fingerprint.json", "scripts/build_spec_bank.py"]
miss = [f for f in need if not os.path.exists(f"{CODE}/{f}")]
print("CODE:", CODE, "| thiếu:", miss or "không")
assert not miss, f"Repo thiếu file: {miss} — đẩy đủ code lên GitHub"

## 4. Cài vLLM và chọn backend chạy được thật trên GPU này
Thử lần lượt (mỗi lần nạp THẬT model trong tiến trình riêng, sinh thử vài token):
`vllm 0.10.1.1` (T4 chạy engine V0 + xformers) TP=2 → TP=1 → `vllm 0.8.5.post1` → backend `hf`.
Lần thử thành công cũng đã tải sẵn model về máy.

In [ ]:
PROBE = f"{WORK}/probe_vllm.py"
open(PROBE, "w").write(r"""
import sys, time
m, tp, maxlen = sys.argv[1], int(sys.argv[2]), int(sys.argv[3])
import torch
from vllm import LLM, SamplingParams
dt = "float16" if torch.cuda.get_device_capability()[0] < 8 else "bfloat16"
t = time.time()
llm = LLM(model=m, dtype=dt, max_model_len=maxlen, tensor_parallel_size=tp, gpu_memory_utilization=0.90,
          enable_prefix_caching=True, seed=0, trust_remote_code=True)
o = llm.generate(["def add(a, b):"], SamplingParams(temperature=0.0, max_tokens=16))
txt = o[0].outputs[0].text
assert txt.strip(), "sinh ra rỗng"
print("PROBE_OK", repr(txt[:60]), f"{time.time() - t:.0f}s")
""")
PROBE_HF = f"{WORK}/probe_hf.py"
open(PROBE_HF, "w").write(r"""
import sys, torch
assert torch.cuda.is_available(), "torch không thấy CUDA"
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
m = sys.argv[1]
tok = AutoTokenizer.from_pretrained(m)
mod = AutoModelForCausalLM.from_pretrained(m, device_map="auto", torch_dtype=torch.float16,
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16))
ids = tok("def add(a, b):", return_tensors="pt").to(mod.device)
print("PROBE_OK", tok.decode(mod.generate(**ids, max_new_tokens=8, do_sample=False)[0]))
""")

def fix_core_libs():
    """Cài vLLM có thể kéo numpy/torch khác -> pandas/sklearn biên dịch theo numpy cũ báo lỗi khi import. Kiểm và sửa."""
    chk = "python -c 'import numpy, pandas, sklearn, openpyxl, yaml; print(numpy.__version__, pandas.__version__, sklearn.__version__)'"
    rc, out = sh(chk, check=False, quiet=True)
    if rc:
        print("pandas/sklearn không hợp numpy mới → cài lại bản hợp")
        sh("pip install -q -U pandas scikit-learn", check=False, quiet=True, timeout=1200)
        rc, out = sh(chk, check=False, quiet=True)
    print("thư viện lõi:", out.strip().splitlines()[-1] if out.strip() else "?", "| OK" if rc == 0 else "| VẪN LỖI")
    return rc == 0

VLLM_ENV = {"VLLM_WORKER_MULTIPROC_METHOD": "spawn", "TOKENIZERS_PARALLELISM": "false"}
BACKEND, TP, RUN_MODEL = None, 1, MODEL
if TEST_MODE or N_GPU == 0:
    BACKEND = "dummy"
else:
    # ghim transformers cùng thời với bản vLLM: transformers mới (5.x) trong image Kaggle có thể không hợp vLLM cũ
    attempts = [("'vllm==0.10.1.1' 'transformers>=4.55,<4.57'", {"VLLM_USE_V1": "0"}),
                ("'vllm==0.8.5.post1' 'transformers>=4.51,<4.53'", {"VLLM_USE_V1": "0"})]
    for spec, extra_env in attempts:
        rc, _ = sh(f"pip install -q {spec} python-calamine openpyxl pyyaml scikit-learn", check=False, quiet=True,
                   timeout=1800)
        if rc:
            print(f"Cài {spec} lỗi → thử phương án sau"); continue
        fix_core_libs()
        for tp in ([2, 1] if N_GPU >= 2 else [1]):
            rc, out = sh(f"python {PROBE} {MODEL} {tp} {MAX_LEN}", check=False, quiet=True, timeout=2400,
                         env={**VLLM_ENV, **extra_env})
            ok = rc == 0 and "PROBE_OK" in out
            print(f"  {spec} TP={tp}: {'CHẠY ĐƯỢC' if ok else 'lỗi'}")
            if ok:
                BACKEND, TP = "vllm", tp
                VLLM_ENV.update(extra_env)
                break
        if BACKEND:
            break
    if BACKEND is None:
        print("!!! vLLM không chạy được trên máy này → backend hf (chậm hơn nhiều; hết giờ thì câu còn lại = đáp án máy tính)")
        sh("pip install -q -U 'transformers>=4.45' accelerate bitsandbytes python-calamine openpyxl pyyaml scikit-learn",
           check=False, quiet=True, timeout=1800)
        rc, out = sh(f"python {PROBE_HF} {MODEL_HF}", check=False, quiet=True, timeout=2400)
        if rc == 0 and "PROBE_OK" in out:
            BACKEND, RUN_MODEL = "hf", MODEL_HF
        else:
            print("!!! Không backend LLM nào chạy được → chỉ dùng đường máy tính (vẫn ra file nộp đủ câu)")
            BACKEND = "none"
os.environ.update(VLLM_ENV)        # các lệnh python chạy sau (pipeline) kế thừa
sh("pip install -q python-calamine openpyxl pyyaml scikit-learn", check=False, quiet=True)
fix_core_libs()
print(f"BACKEND = {BACKEND} | TP = {TP} | MODEL = {RUN_MODEL} | VLLM_USE_V1 = {os.environ.get('VLLM_USE_V1')}")

## 5. Self-test + kho ví dụ đặc tả (dựng từ TRAIN, ~10 phút CPU)

In [ ]:
rc, out = sh(f"python datamate_pipeline.py selftest --split_dir '{TRAIN}'", check=False)
assert "SELFTEST: OK" in out, "Selftest lỗi — xem log ở trên"
SPEC_BANK = f"{CODE}/lexicon/spec_bank.jsonl"
if not (os.path.exists(SPEC_BANK) and os.path.getsize(SPEC_BANK) > 0):
    sh(f"python scripts/build_spec_bank.py --train '{TRAIN}' --cache '{CACHE}/biz' --out '{SPEC_BANK}'")
print("spec_bank:", sum(1 for _ in open(SPEC_BANK, encoding="utf-8")), "ví dụ")

## 6. Lệnh chạy chung

In [ ]:
llm_flag = BACKEND                      # vllm | hf | dummy (chỉ thử cục bộ) | none (không LLM: chỉ đáp án chắc của máy tính)
BATCH = 48 if BACKEND == "vllm" else 4
COMMON = (f"--llm {llm_flag} --model {RUN_MODEL} --tp {TP} --max_model_len {MAX_LEN} --n_paths {N_PATHS} "
          f"--batch {BATCH} --cache '{CACHE}' --spec_bank '{SPEC_BANK}' --lm_texts '{TRAIN}/questions.jsonl' "
          f"--exec_timeout 40 --shed off --provisional on")
print(COMMON)

## 7. (Tuỳ chọn) Đo nhanh trên TRAIN — có điểm

In [ ]:
if RUN_SMOKE and BACKEND != "none":
    t = time.time()
    lim = 3 if TEST_MODE else 8
    sh(f"python datamate_pipeline.py run --split_dir '{TRAIN}' --out '{WORK}/out_smoke' {COMMON} "
       f"--biz biz_101,biz_119 --limit {lim} --fresh", check=False)
    sh(f"python datamate_pipeline.py score --split_dir '{TRAIN}' --pred '{WORK}/out_smoke/submission.jsonl' --only_pred",
       check=False)
    s = json.load(open(f"{WORK}/out_smoke/run_summary.json", encoding="utf-8"))
    if "loi_nap_model" in s:
        print("!!! Pipeline KHÔNG nạp được model dù bước 4 nạp được — xem lỗi:\n", s["loi_nap_model"][-1500:])
    else:
        print("giây/câu:", s.get("giay_moi_cau"), "| lượt gọi model/câu:", s.get("llm_goi_moi_cau"),
              f"| {(time.time() - t) / 60:.1f} phút")

In [ ]:
# (tuỳ chọn) 6 shop TRAIN + fit mô hình bỏ trống theo PA — chỉ dùng TRAIN
ABSTAIN = ""
if RUN_TRAIN_EVAL and BACKEND not in ("none", "dummy"):
    BLIND = "biz_103,biz_108,biz_113,biz_118,biz_124,biz_129"
    budget = max(1800, int((SESSION_HOURS * 0.45 - elapsed_h()) * 3600))
    sh(f"python datamate_pipeline.py run --split_dir '{TRAIN}' --out '{WORK}/out_train' {COMMON} --biz {BLIND} "
       f"--time_budget {budget}", check=False)
    sh(f"python datamate_pipeline.py score --split_dir '{TRAIN}' --pred '{WORK}/out_train/submission.jsonl' --only_pred",
       check=False)
    rc, _ = sh(f"python scripts/fit_abstain.py --train '{TRAIN}' --run '{WORK}/out_train' --out '{WORK}/abstain_model.json'",
               check=False)
    if rc == 0 and os.path.exists(f"{WORK}/abstain_model.json"):
        ABSTAIN = f" --abstain_model '{WORK}/abstain_model.json'"
print("bỏ trống theo mô hình:", ABSTAIN or "không (dùng luật đồng thuận mặc định)")

## 8. Chạy PUBLIC TEST → file nộp
Ngân sách = thời gian còn lại của phiên. Bị dừng giữa chừng thì `submission.jsonl` vẫn đủ câu (câu chưa xong = đáp án chắc chắn của đường máy tính hoặc abstain). Chạy lại ô này sẽ **tiếp tục**, không làm lại.

In [ ]:
OUT_PUB = f"{WORK}/out_public"
budget = int((SESSION_HOURS - elapsed_h()) * 3600) - 15 * 60      # để 15 phút cho kiểm file + nén log
print(f"đã dùng {elapsed_h():.2f} giờ | ngân sách public {budget / 3600:.2f} giờ")
lim = " --limit 3" if TEST_MODE else ""
t = time.time()
sh(f"python datamate_pipeline.py run --split_dir '{PUBLIC}' --out '{OUT_PUB}' {COMMON}{ABSTAIN} "
   f"--time_budget {max(budget, 600)}{lim}", check=False)
print(f"public: {(time.time() - t) / 60:.1f} phút")
if os.path.exists(f"{OUT_PUB}/run_summary.json"):
    print(open(f"{OUT_PUB}/run_summary.json", encoding="utf-8").read()[:1500])

## 9. Kiểm định dạng + ghi file nộp cuối

In [ ]:
VALID = {"number", "percent", "date", "string", "list", "table", "boolean", "unanswerable", "abstain"}   # = VALID_TYPES của pipeline
qids = [json.loads(l)["id"] for l in open(f"{PUBLIC}/questions.jsonl", encoding="utf-8") if l.strip()]
sub, bad = {}, 0
src = f"{OUT_PUB}/submission.jsonl"
if os.path.exists(src):
    for l in open(src, encoding="utf-8"):
        try:
            r = json.loads(l)
        except Exception:
            bad += 1; continue
        if r.get("type") not in VALID or (r["type"] not in ("abstain", "unanswerable") and "answer" not in r):
            bad += 1; continue
        sub[r["id"]] = r
else:
    print("!!! pipeline không tạo được submission → ghi abstain cho mọi câu (xem log bước 8)")
FINAL = f"{WORK}/submission.jsonl"
with open(FINAL, "w", encoding="utf-8") as f:            # đúng thứ tự câu hỏi, mỗi id 1 dòng, thiếu → abstain
    for q in qids:
        r = sub.get(q, {"id": q, "type": "abstain"})
        f.write(json.dumps({"id": q, **{k: v for k, v in r.items() if k != "id"}}, ensure_ascii=False) + "\n")
rows = [json.loads(l) for l in open(FINAL, encoding="utf-8")]
assert len(rows) == len(qids) and len({r["id"] for r in rows}) == len(qids)
assert not any(k.startswith("_") for r in rows for k in r), "lọt khoá nội bộ"
cnt = collections.Counter(r["type"] for r in rows)
print(f"FILE NỘP: {FINAL} | {len(rows)} câu | dòng hỏng đã bỏ: {bad} | câu thiếu → abstain: {len(qids) - len(sub)}")
print("theo loại:", dict(cnt), f"| trả lời {len(rows) - cnt['abstain']}/{len(rows)}")
shutil.copy(FINAL, f"{WORK}/submission_public.jsonl")

In [ ]:
# nén log để tải về (bỏ cache)
dirs = " ".join(d for d in ("out_public", "out_smoke", "out_train") if os.path.isdir(f"{WORK}/{d}"))
sh(f"cd '{WORK}' && rm -f outputs.zip && zip -qr outputs.zip submission.jsonl {dirs} -x '*/cache/*'", check=False)
print(f"Xong sau {elapsed_h():.2f} giờ. Tải về: {WORK}/submission.jsonl (+ outputs.zip)")